# Retrieval of indexed documents based on BM25 via OpenSearch

#### Configuration

In [ ]:
index_name = "trec_robust_2005_bm25"
q = "Information Retrieval Systems"
dataset_name = "aquaint/trec-robust-2005"

In [ ]:
import sys
!{sys.executable} -m pip install -q ir_datasets opensearch-py dotenv

In [ ]:
import pprint

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [ ]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

#### BM25 Search

In [ ]:
def search(query: str, size: int = 10) -> dict:
    body = {
        "size": size,
        "query": {
            "multi_match": {
                "query": query,
                "fields": ["title^2", "text"]  # title gets a boost
            }
        },
    }

    return client.search(index=index_name, body=body)

In [ ]:
resp = search(q, size=5)

print(f"\nTop {len(resp['hits']['hits'])} hits for query: {q}\n")
for hit in resp["hits"]["hits"]:
    src = hit["_source"]
    print(f"[{src['docid']}] {src['title'][:50]}... (score={hit['_score']:.2f})")

#### Search with a Topic from the Dataset

Load the dataset registered under `dataset_name` and run BM25 search for its first topic.

In [ ]:
import importlib
import ir_datasets

# Register the local dataset module (e.g. "ntcir1-adhoc" -> ntcir1_adhoc.py)
dataset_dir = os.path.join(os.getcwd(), '..', 'dataset', dataset_name)
if os.path.isdir(dataset_dir):
    sys.path.append(dataset_dir)
    importlib.import_module(dataset_name.replace('-', '_'))

dataset = ir_datasets.load(dataset_name)

In [ ]:
# Grab the first topic from the dataset
topic = next(dataset.queries_iter())
pprint.pprint(topic)

In [ ]:
topic_query = topic.title
resp = search(topic_query, size=5)

print(f"\nTop {len(resp['hits']['hits'])} hits for topic {topic.query_id}: {topic_query}\n")
for hit in resp["hits"]["hits"]:
    src = hit["_source"]
    print(f"[{src['docid']}] {src['title'][:50]}... (score={hit['_score']:.2f})")